In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Átállítjuk az Ivy cache-t a saját home könyvtáradba, ahol van írási jogod
spark = (
    SparkSession.builder.appName("DataLakeExperiment")
    .master("local[*]")
    .config(
        "spark.jars.ivy", "/home/azureuser/.ivy2"
    )  # <--- Itt a változtatás
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")



:: loading settings :: url = jar:file:/home/azureuser/notebook_env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/azureuser/.ivy2/cache
The jars for the packages stored in: /home/azureuser/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b9fd771f-9ffe-40d1-ac91-bcf605c8592e;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 302ms :: artifacts dl 20ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3  

##Reading

In [21]:


stop_df = spark.read.format("parquet").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/dim_stop.parquet"
)

route_df = spark.read.format("parquet").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/dim_route.parquet"
)

trip_df = spark.read.format("parquet").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/dim_trip.parquet"
)

vhc_positions_df = spark.read.format("delta").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/vehicle_positions"
)

trip_updates_df = spark.read.format("delta").load(
    "/home/azureuser/BKK-Streaming_pipeline/data-lake/bronze/trip_updates"
)



stop_df.show(5)
route_df.show(5)
trip_df.show(5)
vhc_positions_df.show(5)
trip_updates_df.show(5)



+-------+--------------------+---------+---------+---------+-------------------+---------------+
|stop_id|           stop_name| stop_lat| stop_lon|stop_code|wheelchair_boarding|    load_number|
+-------+--------------------+---------+---------+---------+-------------------+---------------+
| 002133|Örs vezér tere M+...|47.500366|  19.1357|   002133|               NULL|20260803_084328|
| 003002|Puskás Ferenc Sta...|47.500368|19.103406|   003002|               NULL|20260803_084328|
| 004716|ÉD metró járműtel...|47.469651| 19.12909|   004716|                2.0|20260803_084328|
| 004948|Metró ÉD járműtel...|47.465239|19.142612|   004948|               NULL|20260803_084328|
| 004952|Metró KNY járműte...|47.502234|19.132179|   004952|               NULL|20260803_084328|
+-------+--------------------+---------+---------+---------+-------------------+---------------+
only showing top 5 rows

+--------+----------------+---------------+----------+--------------------+-----------+---------------

+--------------------+--------------+----------+--------+----------+---------------------+------------------+------------------+-------+---------+---------------------+--------------+-------+-----------------+----------+--------------------+-------------+
|           entity_id|feed_timestamp|   trip_id|route_id|start_date|schedule_relationship|          latitude|         longitude|bearing|    speed|current_stop_sequence|current_status|stop_id|vehicle_timestamp|vehicle_id|       vehicle_label|license_plate|
+--------------------+--------------+----------+--------+----------+---------------------+------------------+------------------+-------+---------+---------------------+--------------+-------+-----------------+----------+--------------------+-------------+
|VehiclePosition-B...|    1785922803|D191891869|    1425|  20260805|            SCHEDULED|47.464508056640625|19.126739501953125|  150.0|      0.0|                    1|    STOPPED_AT| F04017|       1785922793|      1000|  Gloriett-l

+--------------------+--------------+---------+--------+----------+---------------------+-------------+-------+------------+-------------------+--------------+--------------------+
|           entity_id|feed_timestamp|  trip_id|route_id|start_date|schedule_relationship|stop_sequence|stop_id|arrival_time|arrival_uncertainty|departure_time|         ingested_at|
+--------------------+--------------+---------+--------+----------+---------------------+-------------+-------+------------+-------------------+--------------+--------------------+
|TripUpdate-202608...|    1785746430|D09197113|    1380|  20260803|            SCHEDULED|            7| F04284|  1785745505|                 30|    1785745505|2026-08-03 08:41:...|
|TripUpdate-202608...|    1785746430|D09197113|    1380|  20260803|            SCHEDULED|            8| F04285|  1785745625|                 30|    1785745625|2026-08-03 08:41:...|
|TripUpdate-202608...|    1785746430|D09197113|    1380|  20260803|            SCHEDULED|      

##Cleaning

In [ ]:
## Imports
import pyspark.sql.functions as F


count = vhc_positions_df.count() 

## Vehicle cleaning from Null values
clean_vhc_positions_df = vhc_positions_df.filter(
    (F.col("route_id").isNotNull()) & (F.trim(F.col("route_id")) != "") &
    (F.col("trip_id").isNotNull()) & (F.trim(F.col("trip_id")) != "")
)

count_clean = clean_vhc_positions_df.count() 

print(f"{count} count before, {count_clean} count after, {count - count_clean} difference")

## This shows us that every bus has a stop_id in every position
nn = clean_vhc_positions_df.filter(
    ~clean_vhc_positions_df["current_status"].isin("STOPPED_AT", "IN_TRANSIT_TO")
)
nn.show(5)

7980022 count before, 7198051 count after, 781971 difference


+---------+--------------+-------+--------+----------+---------------------+--------+---------+-------+-----+---------------------+--------------+-------+-----------------+----------+-------------+-------------+
|entity_id|feed_timestamp|trip_id|route_id|start_date|schedule_relationship|latitude|longitude|bearing|speed|current_stop_sequence|current_status|stop_id|vehicle_timestamp|vehicle_id|vehicle_label|license_plate|
+---------+--------------+-------+--------+----------+---------------------+--------+---------+-------+-----+---------------------+--------------+-------+-----------------+----------+-------------+-------------+
+---------+--------------+-------+--------+----------+---------------------+--------+---------+-------+-----+---------------------+--------------+-------+-----------------+----------+-------------+-------------+

